# RECIPE GEOFISIKA: ANALISIS MULTICHANNEL ANALYSIS OF SURFACE WAVES (MASW)

**Nama:** Much Fajrial Rahman 

**NIM:** H061231054

**Kelas:** KECERDASAN DAN ARTIFISIAL UNTUK GEOFISIKA (B)

---

## Recipe ID: GEO-MASW-001

**Bidang:** Geofisika near-surface dan geoteknik  
**Topik:** Analisis data gelombang permukaan dengan metode *Multichannel Analysis of Surface Waves* (MASW) untuk memperoleh kurva dispersi gelombang Rayleigh dan estimasi profil kecepatan gelombang geser (\(V_S\)).

> **Petunjuk notebook:** Jalankan sel secara berurutan dari atas ke bawah. Sel Markdown berisi penjelasan, sedangkan sel Code berisi kode Python.

**Catatan penting:** Notebook ini adalah demonstrasi pembelajaran berbasis data sintetik. Data sintetik bukan hasil survei lapangan dan tidak boleh dipakai untuk menyimpulkan kondisi tanah pada lokasi nyata. MASW lapangan umumnya membutuhkan rekaman multichannel, geometri sumber–geofon, pemrosesan dispersi, dan inversi yang tervalidasi.

## 1. Problem — Permasalahan

Metode MASW menggunakan sifat dispersi gelombang permukaan, umumnya gelombang Rayleigh, yang direkam oleh beberapa geofon. Kecepatan fase gelombang permukaan berubah terhadap frekuensi karena gelombang dengan panjang gelombang berbeda memiliki sensitivitas kedalaman yang berbeda. Kurva dispersi kemudian digunakan untuk memperkirakan profil kecepatan gelombang geser (\(V_S\)) bawah permukaan.

Tujuan recipe ini:

1. Membuat atau membaca contoh kurva dispersi.
2. Memvisualisasikan hubungan frekuensi, panjang gelombang, dan kecepatan fase.
3. Menghitung panjang gelombang dari frekuensi dan kecepatan fase.
4. Membuat estimasi awal profil \(V_S\) dari kurva dispersi sintetik menggunakan pendekatan sederhana.
5. Membandingkan kurva dispersi observasi sintetik dengan kurva prediksi pendekatan.
6. Mengekspor data dan ringkasan hasil ke CSV.

**Batasan:** Notebook ini tidak melakukan inversi MASW penuh dari seismogram multichannel. Estimasi \(V_S\) yang ditampilkan adalah ilustrasi sederhana berbasis asumsi, bukan pengganti inversi dispersi Rayleigh yang mempertimbangkan elastisitas, densitas, rasio Poisson, mode gelombang, dan ketidakpastian.

## 2. Minimal Theory — Teori Singkat

Hubungan dasar antara kecepatan fase (\(c\)), frekuensi (\(f\)), dan panjang gelombang (\(\lambda\)) adalah:

\[
\lambda = \frac{c}{f}
\]

Keterangan:
- \(c\): kecepatan fase gelombang permukaan (m/s).
- \(f\): frekuensi (Hz).
- \(\lambda\): panjang gelombang (m).

Gelombang dengan panjang gelombang lebih panjang umumnya menyelidiki bagian bawah permukaan yang lebih dalam daripada gelombang dengan panjang gelombang pendek. Namun, kedalaman investigasi tidak sama persis dengan panjang gelombang dan bergantung pada model serta sensitivitas kernel.

Dalam MASW, kurva dispersi diekstrak dari data multichannel. Kurva ini kemudian diinversi untuk mendapatkan model \(V_S\) terhadap kedalaman. Hubungan antara kecepatan fase Rayleigh dan \(V_S\) bukan hubungan satu-ke-satu yang universal; ia juga dipengaruhi oleh \(V_P\), densitas, rasio Poisson, ketebalan lapisan, dan mode dispersi.

**Catatan metodologis:** Contoh di bawah menggunakan kurva dispersi sintetik yang dibuat untuk demonstrasi dan model \(V_S\) sederhana. Ini bukan simulasi fisika penuh atau inversi standar MASW.

## 3. Persiapan — Library yang Digunakan

Pustaka yang digunakan untuk pengolahan numerik, tabel, dan grafik.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("Library berhasil diimpor.")

## 4. Prompt — Instruksi Analisis untuk AI

> Saya sedang mengolah data MASW. Periksa struktur data dan satuan, visualisasikan kurva dispersi frekuensi–kecepatan fase, hitung panjang gelombang, dan jelaskan rentang sensitivitas kedalaman secara kualitatif. Jika tersedia data seismogram multichannel, jelaskan tahapan pemrosesan untuk menghasilkan citra dispersi dan picking kurva dispersi. Jangan mengarang hasil pengukuran. Bedakan dengan jelas antara data sintetik dan data lapangan.

## 5. Code — Pengaturan Sumber Data

Pilih sumber data:
- `USE_SIMULATED_DATA = True`: membuat kurva dispersi sintetik agar notebook dapat dijalankan tanpa file lapangan.
- `USE_SIMULATED_DATA = False`: membaca `kurva_dispersi_masw.csv`.

Untuk mode data sendiri, CSV harus memiliki kolom `frequency_Hz` dan `phase_velocity_m_per_s`. Data ini merupakan kurva dispersi yang telah diekstrak/picking, bukan rekaman mentah multichannel.

In [ ]:
USE_SIMULATED_DATA = True

if USE_SIMULATED_DATA:
    # Kurva dispersi sintetik untuk demonstrasi, bukan data lapangan.
    rng = np.random.default_rng(7)
    frequency = np.linspace(5, 40, 50)
    # Bentuk kurva contoh: kecepatan fase cenderung meningkat pada frekuensi rendah.
    phase_velocity = 180 + 420 * np.exp(-frequency / 17)
    phase_velocity += rng.normal(0, 5, size=len(frequency))
    phase_velocity = np.clip(phase_velocity, 100, None)

    df = pd.DataFrame({
        "frequency_Hz": frequency,
        "phase_velocity_m_per_s": phase_velocity
    })
    print("Menggunakan DATA SINTETIK. Ini bukan hasil pengukuran lapangan.")
else:
    input_path = Path("kurva_dispersi_masw.csv")
    if not input_path.exists():
        raise FileNotFoundError(
            "File kurva_dispersi_masw.csv tidak ditemukan. Letakkan file di folder notebook."
        )
    df = pd.read_csv(input_path)
    print(f"Membaca data dari: {input_path.resolve()}")

df.head()

## 6. Code — Pemeriksaan Dasar Data

Memeriksa kolom wajib, nilai kosong, rentang frekuensi, dan kecepatan fase yang bernilai positif.

In [ ]:
required = ["frequency_Hz", "phase_velocity_m_per_s"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing}")

df = df[required].copy()
for col in required:
    df[col] = pd.to_numeric(df[col], errors="coerce")

if df.isna().any().any():
    raise ValueError("Data mengandung nilai kosong atau non-numerik.")
if len(df) < 5:
    raise ValueError("Kurva dispersi memerlukan setidaknya 5 titik untuk demonstrasi.")
if (df["frequency_Hz"] <= 0).any():
    raise ValueError("Frekuensi harus lebih besar dari nol.")
if (df["phase_velocity_m_per_s"] <= 0).any():
    raise ValueError("Kecepatan fase harus lebih besar dari nol.")

df = df.sort_values("frequency_Hz").reset_index(drop=True)
if df["frequency_Hz"].duplicated().any():
    print("Peringatan: ada frekuensi duplikat. Periksa hasil picking kurva dispersi.")

print("Jumlah titik:", len(df))
print(f"Rentang frekuensi: {df.frequency_Hz.min():.2f}–{df.frequency_Hz.max():.2f} Hz")
print(f"Rentang kecepatan fase: {df.phase_velocity_m_per_s.min():.2f}–{df.phase_velocity_m_per_s.max():.2f} m/s")
display(df.head())

## 7. Code — Visualisasi Kurva Dispersi

Kurva dispersi menampilkan kecepatan fase terhadap frekuensi. Pastikan sumbu dan satuan sesuai dengan format keluaran perangkat lunak yang digunakan.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(df["frequency_Hz"], df["phase_velocity_m_per_s"], "o-", label="Kurva dispersi")
plt.xlabel("Frekuensi (Hz)")
plt.ylabel("Kecepatan fase (m/s)")
plt.title("Kurva Dispersi MASW")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 8. Code — Menghitung Panjang Gelombang

Panjang gelombang dihitung menggunakan \(\lambda=c/f\). Nilai ini membantu melihat rentang skala panjang gelombang yang terwakili oleh kurva dispersi, tetapi tidak langsung sama dengan kedalaman investigasi.

In [ ]:
df["wavelength_m"] = (
    df["phase_velocity_m_per_s"] / df["frequency_Hz"]
)

display(df.head())

plt.figure(figsize=(9, 5))
plt.plot(df["frequency_Hz"], df["wavelength_m"], "o-")
plt.xlabel("Frekuensi (Hz)")
plt.ylabel("Panjang gelombang (m)")
plt.title("Panjang Gelombang terhadap Frekuensi")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Rentang panjang gelombang: {df.wavelength_m.min():.2f}–{df.wavelength_m.max():.2f} m")

## 9. Demonstrasi — Model \(V_S\) Sederhana

Bagian ini membuat model lapisan ilustratif dan perkiraan kecepatan fase menggunakan faktor pendekatan tetap. Ini **bukan inversi MASW fisika penuh**. Faktor pendekatan tersebut hanya digunakan agar mahasiswa dapat memahami bahwa kurva dispersi berkaitan dengan kecepatan geser dan kedalaman.

Dalam analisis MASW sebenarnya, inversi harus menggunakan solver dispersi gelombang Rayleigh dan menyesuaikan parameter seperti \(V_S\), \(V_P\), densitas, ketebalan lapisan, serta pilihan mode dispersi. Jangan menganggap nilai \(V_S\) dari demonstrasi ini sebagai hasil kuantitatif yang valid.

In [ ]:
# Model lapisan ilustratif saja, bukan hasil inversi.
layer_model = pd.DataFrame({
    "layer": [1, 2, 3],
    "thickness_m": [3.0, 7.0, np.nan],  # NaN = half-space
    "Vs_m_per_s": [180.0, 320.0, 550.0]
})
display(layer_model)

plt.figure(figsize=(5, 6))
depth_top = 0.0
for _, row in layer_model.iterrows():
    thickness = row["thickness_m"]
    if np.isnan(thickness):
        plt.plot([row["Vs_m_per_s"], row["Vs_m_per_s"]], [depth_top, depth_top + 5], linewidth=8)
        plt.text(row["Vs_m_per_s"] + 8, depth_top + 2, f"Vs={row['Vs_m_per_s']:.0f} m/s")
    else:
        plt.plot([row["Vs_m_per_s"], row["Vs_m_per_s"]], [depth_top, depth_top + thickness], linewidth=8)
        plt.text(row["Vs_m_per_s"] + 8, depth_top + thickness / 2, f"Vs={row['Vs_m_per_s']:.0f} m/s")
        depth_top += thickness

plt.gca().invert_yaxis()
plt.xlabel("Vs (m/s)")
plt.ylabel("Kedalaman (m; skematik)")
plt.title("Model Vs Lapisan Ilustratif")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Interpretasi Kualitatif

Secara umum, frekuensi lebih tinggi berkaitan dengan panjang gelombang yang lebih pendek dan sensitivitas yang relatif dangkal, sedangkan frekuensi lebih rendah berkaitan dengan panjang gelombang yang lebih panjang dan sensitivitas yang relatif lebih dalam. Namun, hubungan ini tidak boleh diubah menjadi kedalaman pasti tanpa analisis sensitivitas atau inversi.

Hal yang perlu diperhatikan:
- Apakah kurva dispersi halus dan konsisten?
- Apakah ada titik picking yang menyimpang dari tren utama?
- Apakah mode fundamental dan mode lebih tinggi telah dibedakan?
- Apakah rentang frekuensi cukup untuk target kedalaman?
- Apakah model inversi cocok dengan kurva dispersi teramati dan memiliki ketidakpastian yang dilaporkan?

**Penting:** Notebook ini hanya mengolah kurva dispersi yang sudah tersedia. Ia tidak mengekstrak kurva dispersi langsung dari data seismogram multichannel mentah.

## 11. Code — Ekspor Hasil ke CSV

Menyimpan kurva dispersi dengan panjang gelombang dan model lapisan ilustratif.

In [ ]:
df.to_csv("hasil_kurva_dispersi_masw.csv", index=False)
layer_model.to_csv("model_vs_ilustratif.csv", index=False)

summary = pd.DataFrame([{
    "data_type": "sintetik" if USE_SIMULATED_DATA else "kurva dispersi input",
    "number_of_dispersion_points": len(df),
    "frequency_min_Hz": df["frequency_Hz"].min(),
    "frequency_max_Hz": df["frequency_Hz"].max(),
    "phase_velocity_min_m_per_s": df["phase_velocity_m_per_s"].min(),
    "phase_velocity_max_m_per_s": df["phase_velocity_m_per_s"].max(),
    "wavelength_min_m": df["wavelength_m"].min(),
    "wavelength_max_m": df["wavelength_m"].max(),
    "note": "Model Vs hanya ilustrasi; bukan hasil inversi MASW penuh."
}])
summary.to_csv("ringkasan_masw.csv", index=False)

print("File tersimpan:")
print("- hasil_kurva_dispersi_masw.csv")
print("- model_vs_ilustratif.csv")
print("- ringkasan_masw.csv")
display(summary)

## 12. Optional Check — Checklist Kualitas

- Data yang dipakai dibedakan antara sintetik dan lapangan.
- Satuan frekuensi dan kecepatan fase telah diperiksa.
- Kurva dispersi berasal dari picking yang sudah diperiksa.
- Mode dispersi telah diidentifikasi dengan hati-hati.
- Rentang frekuensi sesuai dengan target investigasi.
- Panjang gelombang tidak disamakan secara langsung dengan kedalaman investigasi.
- Model \(V_S\) diberi label sebagai ilustrasi jika tidak melalui inversi penuh.
- Hasil inversi lapangan divalidasi terhadap kurva dispersi, geologi, atau data geofisika lain jika tersedia.

## 13. Kesimpulan

Recipe GEO-MASW-001 memperkenalkan alur dasar analisis kurva dispersi MASW, mulai dari pemeriksaan data, visualisasi frekuensi terhadap kecepatan fase, perhitungan panjang gelombang, pengenalan model \(V_S\) berlapis, hingga ekspor hasil ke CSV. Notebook ini ditujukan untuk pembelajaran dan belum menggantikan workflow MASW lengkap yang memproses rekaman multichannel, mengekstrak kurva dispersi, lalu melakukan inversi dengan solver yang sesuai.